In [ ]:
# Setup: find the course materials and load our helper code. Run this cell first.
import subprocess
import sys
from pathlib import Path

# Colab has a module called google.colab; a laptop does not.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: download the course materials once.
    TASK_DIR = Path("/content/intern-training/tasks/esm2-enzyme-transfer")
    if not TASK_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "esm2-v1",
             "https://github.com/lagillenwater/intern-training", "/content/intern-training"],
            check=True,
        )
    # Install the exact package versions this course was tested with.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 'accelerate==1.16.0', 'transformers==5.23.0', 'umap-learn==0.6.0'],
        check=True,
    )
else:
    # On a laptop: walk up from the current folder until we find the folder
    # that holds both helpers/ and checks.py.
    TASK_DIR = None
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "helpers").is_dir() and (folder / "checks.py").exists():
            TASK_DIR = folder
            break
    if TASK_DIR is None:
        raise RuntimeError("Could not find the course materials (a folder with helpers/ and checks.py).")

# Make our helper code importable, then load it.
sys.path.insert(0, str(TASK_DIR))
from helpers import *
from checks import *

print(f"Setup complete. Materials: {TASK_DIR}")


# 05 Interpret the results

**What this step is for:** look for the reasons behind your scores. You draw UMAP pictures of how ESM-2 sees the test proteins before and after fine-tuning, look up proteins the fine-tuned model got wrong on UniProt, and answer six questions in your own words. Your final presentation draws on this notebook and on notebook 04.

**Time:** about 1.5 hours.

**Reads:** `data/enzymes.tsv`; `embeddings_base.npz` from notebook 02; `model_finetuned/` and `predictions_finetuned.tsv` from notebook 03; `paired_differences.tsv`, `per_class_f1.tsv`, and `figures/distance_curve.png` from notebook 04. **Writes:** `embeddings_finetuned.npz` and four UMAP figures, `figures/umap_before_species.png`, `figures/umap_before_class.png`, `figures/umap_after_species.png`, and `figures/umap_after_class.png`, in your run folder on Google Drive.

**Before you start:** use a GPU runtime (in Colab: Runtime > Change runtime type > T4 GPU). One cell runs your fine-tuned model over every test protein, which is slow without a GPU. Then run the Setup cell at the top and work top to bottom. If a cell below a `TODO` shows an error, first make sure that `TODO` passed its check.

In [ ]:
# Configuration for this run (the same in every notebook).
RUN_NAME = "esm2_8M"                      # a label for this run: the small ESM-2 model, with 8 million parameters
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"  # the model's name on Hugging Face, the website it is downloaded from
WORK_DIR = setup_workdir(RUN_NAME)        # reconnects Google Drive on Colab and returns the run folder
print(WORK_DIR)

In [ ]:
# The standard data tools (notebook 00, Part 4).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = load_dataset()                  # the cleaned table: one row per enzyme, all five species
test_df = df[df["split"] == "test"]  # every test protein, all five species (as in notebook 03)
print(f"{len(test_df):,} test proteins")
test_df["species"].value_counts().loc[SPECIES_ORDER]

## Embeddings before and after fine-tuning
You need two embeddings for every test protein: one from ESM-2 before fine-tuning and one after.

**Before:** notebook 02 saved the frozen ESM-2 embeddings of every protein in `df`, in `embeddings_base.npz`. An `.npz` file holds several named numpy arrays: here `accessions` (the protein IDs), `embeddings` (one row of 320 numbers per protein), and `model` (the name of the model that made them). The next cell picks out the rows of the test proteins by matching their accessions.

In [ ]:
base_path = require_file(WORK_DIR / "embeddings_base.npz", made_by="Notebook 02 (baseline)")
# "with" opens the file and closes it again at the end of the block, like Java's try-with-resources.
with np.load(base_path) as saved:
    base_accessions = saved["accessions"]
    base_embeddings = saved["embeddings"]
    print(f"Saved embeddings were made with {saved['model']}; this run uses {MODEL_NAME}")

# A Series that maps each accession to its row number in the saved arrays, like a Java HashMap<String, Integer>.
row_of = pd.Series(np.arange(len(base_accessions)), index=base_accessions)
# Look up every test accession at once (no loop), then take those rows of the embeddings.
test_rows = row_of.loc[test_df["accession"]].to_numpy()
E_base = base_embeddings[test_rows]
print(E_base.shape)  # (number of test proteins, 320)

**After:** the fine-tuned model from notebook 03 has new weights inside ESM-2. The next cell runs it over every test protein and averages over the amino acids (mean pooling), exactly as notebook 02 did with the frozen model. Because both embeddings are made the same way, any difference between the two pictures comes from fine-tuning.

The first time, this takes a few minutes on a GPU; then it saves the result to your Drive as `embeddings_finetuned.npz`, and later runs load it in seconds.

In [ ]:
# Stop with a clear message if notebook 03 has not saved the model yet.
require_file(WORK_DIR / "model_finetuned", made_by="Notebook 03 (fine-tune)")
# cached_embeddings is the helper from notebook 02. This time it uses your fine-tuned model, given by its folder.
E_ft = cached_embeddings(test_df, WORK_DIR / "embeddings_finetuned.npz", str(WORK_DIR / "model_finetuned"))
print(E_ft.shape)  # one row per test protein, in the same order as test_df and E_base

## UMAP: a flat picture of the embeddings
You presented on dimension reduction. **UMAP** takes each protein's 320 numbers and finds a spot for it on a flat page, so that proteins with similar embeddings land near each other. Each protein becomes one dot.

How to read a UMAP plot:
- Dots that are close together have similar embeddings: the model sees those proteins as alike.
- A tight, separate cluster is a group of proteins that the model sees as alike and as different from the rest.
- The axes have no units, and the distance between two clusters that are far apart means little. Read who is next to whom, not exact distances.

UMAP starts from a random layout. The `seed` fixes that randomness, so the same seed and the same embeddings always give the same picture. You will use the same seed, `0`, for both, so the two pictures are made the same way.

**Prediction (write this before you run the next cell):** Before fine-tuning, do you think the test proteins will group by organism (human proteins near human proteins, yeast near yeast) or by function (enzymes of the same class near each other)? Why?

In [ ]:
# TODO: Squash both sets of embeddings to 2 dimensions with UMAP: coords_before from E_base, and coords_after from E_ft.
#       Use seed=0 for both. Each one takes a minute or two.
# Hint: compute_umap(embeddings, seed=0) returns one row with an x and a y for each protein.
coords_before = None  # replace None with your code
coords_after = None  # replace None with your code

In [ ]:
check_umap_coords(coords_before, len(test_df), name="coords_before");
check_umap_coords(coords_after, len(test_df), name="coords_after");

## Plot the UMAPs
Our helper `plot_umap(coords, labels, title)` draws one dot per protein and colors the dots by `labels`, which needs one label per protein, in the same order as `coords`:
- `test_df["species"]` colors each dot by its species.
- `test_df["ec_class"]` colors each dot by its enzyme class; the legend shows the class names.

Like the plots in notebook 04, it returns a figure that you save with `.savefig(...)`. When one cell makes two figures, the notebook shows both.

In [ ]:
# TODO: Plot coords_before twice: once colored by species and once colored by enzyme class, each with a title.
#       Save them in WORK_DIR / "figures" as umap_before_species.png and umap_before_class.png.
# Hint: plot_umap(coords, labels, title) returns the figure; save each one with .savefig(path, dpi=150, bbox_inches="tight").
# YOUR CODE HERE

**Compare:** Was your prediction right? What surprised you?

In [ ]:
# TODO: Make the same two plots for coords_after, the fine-tuned embeddings.
#       Save them as umap_after_species.png and umap_after_class.png in WORK_DIR / "figures".
# Hint: the same plot_umap and .savefig calls as in your cell above, with coords_after, new titles, and new file names.
# YOUR CODE HERE

Question 4 below asks what changed between the before and after pictures.

## Proteins the model got wrong
Scores tell you how often the model is wrong; looking at single proteins can tell you why. You will make a table of the yeast and *E. coli* proteins that the fine-tuned model got wrong, with a link to each protein's page on UniProt, the protein database our data came from.

Two new pandas tools help:
- `column.isin(["a", "b"])` is True for the rows whose value is in the list. It is a shorter way to write `(column == "a") | (column == "b")`.
- Adding a string to a column of text, `"start" + column`, glues the string to the front of every value, row by row, with no loop.

In [ ]:
# The fine-tuned model's predictions for every test protein, saved by notebook 03.
ft_path = require_file(WORK_DIR / "predictions_finetuned.tsv", made_by="Notebook 03 (fine-tune)")
ft_preds = pd.read_csv(ft_path, sep="\t")  # columns: accession, species, y_true, y_pred_finetuned
ft_preds.head()

In [ ]:
# TODO: Make a table named misclassified with the rows of ft_preds for yeast and E. coli proteins that the fine-tuned
#       model got wrong (y_true is not equal to y_pred_finetuned). Then add a column uniprot_url holding each protein's
#       web address: "https://www.uniprot.org/uniprotkb/" followed by its accession.
# Hint: combine an .isin(["yeast", "ecoli"]) mask on the species column and a != mask between y_true and y_pred_finetuned
#       with &, and keep those rows with .copy(); then set misclassified["uniprot_url"] to the address text + the accession column.
misclassified = None  # replace None with your code

In [ ]:
check_misclassified(misclassified, ft_preds);

In [ ]:
# Add each protein's gene, name, EC number, and length from df, matching rows on accession.
lookup = misclassified.merge(df[["accession", "gene", "protein_name", "ec", "length"]], on="accession")
# .map(CLASS_NAMES) replaces each class number with its name, using the dict as a lookup table.
lookup["true_class"] = lookup["y_true"].map(CLASS_NAMES)
lookup["predicted_class"] = lookup["y_pred_finetuned"].map(CLASS_NAMES)
print(f"{len(lookup):,} misclassified proteins:", lookup["species"].value_counts().to_dict())

# .sample(15, random_state=0) picks 15 rows at random. Change the 0 to another number to see 15 different ones.
lookup.sample(15, random_state=0)[
    ["species", "gene", "protein_name", "ec", "length", "true_class", "predicted_class", "uniprot_url"]
]

Question 5 below asks you to look up two or three of these proteins. To open one, copy its `uniprot_url` into a new browser tab.

## Interpretation questions
Answer each question in your own words, in the cell under it. Use numbers from your tables and point to your figures. These answers are the core of your presentation. The next cell shows your headline figure and paired differences from notebook 04 again, for Question 1.

In [ ]:
# display() and Image() show a saved picture inside the notebook.
from IPython.display import Image, display

display(Image(filename=str(require_file(WORK_DIR / "figures" / "distance_curve.png", made_by="Notebook 04 (evaluate)"))))
diffs = pd.read_csv(require_file(WORK_DIR / "paired_differences.tsv", made_by="Notebook 04 (evaluate)"), sep="\t")
diffs.round(3)

**Question 1. Distance.** How does performance change as the species get further from human? Does the gain from fine-tuning grow or shrink with distance? Support your answer with numbers from the figure and the table.

**Your answer:**

**Question 2. Mouse.** Why might the mouse scores be so high? Is mouse a fair test of generalization, that is, of whether the model can recognize the class of enzymes that are unlike the ones it learned from?

**Your answer:**

In [ ]:
# The fine-tuned model's F1 for each class (rows) in each species (columns), from notebook 04, for Question 3.
# Read across a row to follow one class from human to E. coli. NaN means that species has no test proteins of that class.
class_f1 = pd.read_csv(require_file(WORK_DIR / "per_class_f1.tsv", made_by="Notebook 04 (evaluate)"), sep="\t")
finetuned_f1 = class_f1[class_f1["model"] == "finetuned"]
finetuned_f1.pivot(index="class_name", columns="species", values="f1").reindex(
    index=list(CLASS_NAMES.values()), columns=SPECIES_ORDER
).round(2)

**Question 3. Classes.** Which enzyme classes transfer best from human to other species, and which transfer worst, especially to *E. coli*? Pick the best and the worst class for *E. coli*, and say what kind of job the enzymes in each do (notebook 01 has a short description of each class). Why might one kind of job be easier to recognize than another in a species this far from human? Think about which jobs every living cell needs, and which jobs a bacterium might do differently from an animal cell.

**Your answer:**

**Question 4. UMAP.** Look at your four UMAP plots. Before fine-tuning, are the proteins grouped by organism or by function? What changed after fine-tuning? Why would fine-tuning change the picture in that way?

**Your answer:**

**Question 5. Proteins the model got wrong.** Pick two or three yeast or *E. coli* enzymes from your `misclassified` table. Open each one's `uniprot_url` and read its page. For each protein, write its accession and name, its true and predicted class, what you found, and a reason the model might have gotten it wrong.

Places to look on a UniProt page:
- **Function**: what the enzyme does, and its EC number or numbers.
- **Family & Domains**: which protein family it belongs to, and its domains (the separately folded parts of the protein).
- **Sequence**: its length. The model reads only the first 1,022 amino acids.

**Your answer:**

**Question 6. One open item.** Choose one thing to find out on your own, and add code cells under this one for it. It can be an extra plot that tests an idea from your answers above, or a change to one training setting (the learning rate or the number of epochs) with a before-and-after comparison. Write down what you did, what you expected, and what happened.

A training change means running notebook 03 again, which takes about as long as the first time. Plan it with your mentor first, so the new model is saved in a new run folder and does not replace your main results.

**Your answer:**

## Done
Your UMAP figures are saved in your run folder, next to the headline figure from notebook 04.

Next: your presentation (the README suggests an outline). **Check-in 3** is the question time after it. Your mentor will ask, without AI help, about evolutionary distance, the mouse result, which classes transferred, and your UMAP plots, so reread your answers to Questions 1 to 4 first. Then open your pull request.

## Stretch goals (optional)
The sections below are for if you finish everything above early; try them in order.